# Cell 1: download your repo

In [1]:
!rm -rf /kaggle/working/Gemma4_FineTune_Creativity
!rm -rf /kaggle/working/Gemma4_FineTune_Creativity-main
!rm -f /kaggle/working/repo.zip

!wget -qO /kaggle/working/repo.zip https://github.com/AlinBolcas/Gemma4_FineTune_Creativity/archive/refs/heads/main.zip
!unzip -q /kaggle/working/repo.zip -d /kaggle/working
!rm -f /kaggle/working/repo.zip
!mv /kaggle/working/Gemma4_FineTune_Creativity-main /kaggle/working/Gemma4_FineTune_Creativity

# Cell 2: move into repo

In [2]:
import os, sys
from pathlib import Path

ROOT = Path("/kaggle/working/Gemma4_FineTune_Creativity")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(ROOT)

/kaggle/working/Gemma4_FineTune_Creativity


# Cell 3: install training deps

In [3]:
!pip install -q -U unsloth datasets trl peft accelerate bitsandbytes huggingface_hub sentencepiece
!pip install -q "protobuf>=5.26,<6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 29.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.6 MB/s eta 0:00:00:00:010:01
 

# Cell 4: load Hugging Face token

In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
os.environ["HUGGINGFACE_ACCESS_TOKEN"] = hf_token

print("HF token loaded")

HF token loaded


# Cell 5: verify GPU

In [5]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


# Cell 6: build a fresh Kaggle config
This makes a new config using Kaggle paths, not your Mac paths.

In [6]:
from src.III_fineTune.sft_train import discover_dataset_bundles

bundles = discover_dataset_bundles()
for b in bundles:
    print(
        b["base"],
        "train=", b["train_count"],
        "eval=", b["eval_count"],
        "test=", b["test_count"],
    )

bundles

all_domains_augmented_20260417_155341 train= 334 eval= 18 test= 20


[{'base': 'all_domains_augmented_20260417_155341',
  'combined_path': PosixPath('/kaggle/working/Gemma4_FineTune_Creativity/data/input/all_domains_augmented_20260417_155341_sft.jsonl'),
  'train_path': PosixPath('/kaggle/working/Gemma4_FineTune_Creativity/data/input/train/all_domains_augmented_20260417_155341_train.jsonl'),
  'eval_path': PosixPath('/kaggle/working/Gemma4_FineTune_Creativity/data/input/eval/all_domains_augmented_20260417_155341_eval.jsonl'),
  'test_path': PosixPath('/kaggle/working/Gemma4_FineTune_Creativity/data/input/test/all_domains_augmented_20260417_155341_test.jsonl'),
  'train_count': 334,
  'eval_count': 18,
  'test_count': 20}]

In [ ]:
from src.III_fineTune.sft_train import (
    discover_dataset_bundles,
    build_run_config,
    save_run_config,
    run_preflight,
)

bundles = discover_dataset_bundles()
if not bundles:
    raise RuntimeError("No dataset bundles found. Run format_sft first.")

bundle = sorted(bundles, key=lambda b: b["train_count"], reverse=True)[0]
train_count = int(bundle["train_count"])
eval_count = int(bundle["eval_count"])
test_count = int(bundle["test_count"])

print("Using bundle:", bundle["base"])
print("train / eval / test:", train_count, eval_count, test_count)

config = build_run_config(
    bundle=bundle,
    model_alias="e4b",
    backend="unsloth",
    run_name=f"{bundle['base']}_e4b_v3_obvious",
)

# --- Runtime ---
config["runtime"]["target"] = "kaggle"
config["runtime"]["post_train_eval"] = True
config["runtime"]["preflight_sample_count"] = 100  # better max_seq_length safety check

# --- Output ---
config["training"]["output_dir"] = f"data/output/models/{bundle['base']}_e4b_v3_obvious"

# --- Batch / sequence ---
config["training"]["max_steps"] = 0
config["training"]["per_device_train_batch_size"] = 1
config["training"]["gradient_accumulation_steps"] = 4
config["training"]["max_seq_length"] = 1536  # less truncation risk than 1024

# --- LoRA: high capacity (more visible behavior change) ---
config["training"]["lora_r"] = 64
config["training"]["lora_alpha"] = 128
config["training"]["lora_dropout"] = 0.0

# --- Regularization: removed (more memorization / stronger style fit) ---
config["training"]["weight_decay"] = 0.0

# Strong but less explosive than your 3e-4 + huge epoch schedule
config["training"]["num_train_epochs"] = 6
config["training"]["learning_rate"] = 2e-4
config["training"]["warmup_steps"] = 50

# Tiny helper print (mental model: effective batch ~= 4 on 1 GPU)
steps_per_epoch = train_count / (config["training"]["per_device_train_batch_size"] * config["training"]["gradient_accumulation_steps"])
print("Approx optimizer steps:", int(steps_per_epoch * config["training"]["num_train_epochs"]))

config_path = save_run_config(config)
print("Saved config:", config_path)

summary = run_preflight(config)
summary

Using bundle: all_domains_augmented_20260417_155341
train / eval / test: 334 18 20
Saved config: /kaggle/working/Gemma4_FineTune_Creativity/src/III_fineTune/configs/runs/all_domains_augmented_20260417_155341_e4b_v2_strong.json


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).



  Local Preflight
  run_name: all_domains_augmented_20260417_155341_e4b_v2_strong
  model:    google/gemma-4-E4B-it
  backend:  unsloth
  train:    334
  eval:     18
  test:     20


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]


Token lengths:
  min=652  avg=694.2  max=739
  OK: sample fits max_seq_length

Training row style:
  system rows in sample: 5/5
  thinking flag in config: False
  chat template: gemma-4

Saved preflight: /kaggle/working/Gemma4_FineTune_Creativity/data/output/training_runs/preflight/all_domains_augmented_20260417_155341_e4b_v2_strong_preflight.json


{'run_name': 'all_domains_augmented_20260417_155341_e4b_v2_strong',
 'model': 'google/gemma-4-E4B-it',
 'backend': 'unsloth',
 'train_count': 334,
 'eval_count': 18,
 'test_count': 20,
 'sample_count': 5,
 'sample_token_min': 652,
 'sample_token_avg': 694.2,
 'sample_token_max': 739,
 'max_seq_length': 1024,
 'system_prompt_rows_in_sample': 5}

# Cell 7: train

In [8]:
from src.III_fineTune.sft_train import train_from_config
train_from_config(config)

/kaggle/working/Gemma4_FineTune_Creativity/src/III_fineTune/sft_train.py:403: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

Loading unsloth/gemma-4-E4B-it with Unsloth...
==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

...


Starting Unsloth training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 334 | Num Epochs = 8 | Total steps = 672
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 73,400,320 of 8,069,556,768 (0.91% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,10.536036
2,10.382188
3,10.708819
4,10.307344
5,9.970772
6,8.179390
7,6.887410
8,6.155344
9,5.335568
10,5.019053


Saved model: /kaggle/working/Gemma4_FineTune_Creativity/data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong
Post-train eval skipped: CUDA out of memory. Tried to allocate 1.51 GiB. GPU 0 has a total capacity of 14.56 GiB of which 825.81 MiB is free. Including non-PyTorch memory, this process has 13.75 GiB memory in use. Of the allocated memory 13.44 GiB is allocated by PyTorch, and 159.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Saved stats: /kaggle/working/Gemma4_FineTune_Creativity/data/output/training_runs/all_domains_augmented_20260417_155341_e4b_v2_strong


# ZIP Fine Tuned Adaptor

In [ ]:
run_name = config['run_name']
!cd /kaggle/working/Gemma4_FineTune_Creativity && zip -r /kaggle/working/{run_name}_adapter.zip data/output/models/{run_name}
!cd /kaggle/working/Gemma4_FineTune_Creativity && zip -r /kaggle/working/{run_name}_stats.zip data/output/training_runs/{run_name}

  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/ (stored 0%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/README.md (deflated 46%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/tokenizer_config.json (deflated 73%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/tokenizer.json (deflated 83%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/checkpoint-500/ (stored 0%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/checkpoint-500/README.md (deflated 65%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/checkpoint-500/tokenizer_config.json (deflated 73%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_strong/checkpoint-500/trainer_state.json (deflated 81%)
  adding: data/output/models/all_domains_augmented_20260417_155341_e4b_v2_stro